In [3]:
import pandas as pd
# 1. Load all three CSV files.
customers = pd.read_csv('data/customers.csv')
products = pd.read_csv('data/products.csv')
orders = pd.read_csv('data/orders.csv')

# 2. Display the dimensions of each DataFrame.
print('Customers:', customers.shape)
print('Products:', products.shape)
print('Orders:', orders.shape)

# 3. Identify the columns in each dataset.
print('Customers columns:', list(customers.columns))
print('Products columns:', list(products.columns))
print('Orders columns:', list(orders.columns))

# 4. Identify the common columns between datasets.
customers_cols = set(customers.columns)
products_cols = set(products.columns)
orders_cols = set(orders.columns)

print('Common between customers & orders:', customers_cols & orders_cols)
print('Common between products & orders:', products_cols & orders_cols)
print('Common between customers & products:', customers_cols & products_cols)

Customers: (3000, 5)
Products: (100, 4)
Orders: (15000, 5)
Customers columns: ['Customer_ID', 'Customer_Name', 'City', 'Age', 'Gender']
Products columns: ['Product_ID', 'Product_Name', 'Category', 'Price']
Orders columns: ['Order_ID', 'Customer_ID', 'Product_ID', 'Quantity', 'Order_Date']
Common between customers & orders: {'Customer_ID'}
Common between products & orders: {'Product_ID'}
Common between customers & products: set()


In [7]:
# 6. Check each DataFrame for missing values.
print(customers.isnull().sum())
print(products.isnull().sum())
print(orders.isnull().sum())

# 7. Check for duplicate records.
print('Customers duplicates:', customers.duplicated().sum())
print('Products duplicates:', products.duplicated().sum())
print('Orders duplicates:', orders.duplicated().sum())

# 8. Check the validity of Customer_ID and Product_ID.
valid_customer_ids = set(customers['Customer_ID'])
valid_product_ids = set(products['Product_ID'])

invalid_customer_refs = orders[~orders['Customer_ID'].isin(valid_customer_ids)]
invalid_product_refs = orders[~orders['Product_ID'].isin(valid_product_ids)]

print('Invalid Customer_ID references:', len(invalid_customer_refs))
print('Invalid Product_ID references:', len(invalid_product_refs))

# 9. Convert Order_Date into datetime.
orders['Order_Date'] = pd.to_datetime(orders['Order_Date'])

print(orders['Order_Date'].dtype)
print(orders['Order_Date'].min(), 'to', orders['Order_Date'].max())

# 10. Check whether every order corresponds to a valid customer.
orders_valid_customer = orders['Customer_ID'].isin(valid_customer_ids)

print('All orders have valid Customer_ID:', orders_valid_customer.all())

# 11. Check whether every order corresponds to a valid product.
orders_valid_product = orders['Product_ID'].isin(valid_product_ids)
 
print('All orders have valid Product_ID:', orders_valid_product.all())


Customer_ID      0
Customer_Name    0
City             0
Age              0
Gender           0
dtype: int64
Product_ID      0
Product_Name    0
Category        0
Price           0
dtype: int64
Order_ID       0
Customer_ID    0
Product_ID     0
Quantity       0
Order_Date     0
dtype: int64
Customers duplicates: 0
Products duplicates: 0
Orders duplicates: 0
Invalid Customer_ID references: 0
Invalid Product_ID references: 0
datetime64[us]
2025-01-01 00:00:00 to 2028-06-03 22:00:00
All orders have valid Customer_ID: True
All orders have valid Product_ID: True


In [9]:
# 12. Merge orders with customers.
merged = orders.merge(customers, on='Customer_ID', how='inner')

print('Merged shape:', merged.shape)
print(merged.head())

# 13. Merge the resulting DataFrame with products.
merged = orders.merge(customers, on='Customer_ID', how='inner')
merged = merged.merge(products, on='Product_ID', how='inner')

print('Final shape:', merged.shape)
print(merged.columns.tolist())

# 14. Create a final consolidated DataFrame.
# The final DataFrame should contain information such as:
# Order_ID
# Customer_ID
# Customer_Name
# City
# Age
# Gender
# Product_ID
# Product_Name
# Category
# Price
# Quantity
# Order_Date
merged = orders.merge(customers, on='Customer_ID', how='inner')
merged = merged.merge(products, on='Product_ID', how='inner')

final_df = merged[['Order_ID', 'Customer_ID', 'Customer_Name', 'City', 'Age', 'Gender',
                    'Product_ID', 'Product_Name', 'Category', 'Price', 'Quantity', 'Order_Date']]

print(final_df.shape)
print(final_df.columns.tolist())

Merged shape: (15000, 9)
  Order_ID Customer_ID Product_ID  Quantity          Order_Date  \
0  O000001      C00861      P0027         4 2025-01-01 00:00:00   
1  O000002      C01295      P0047         3 2025-01-01 02:00:00   
2  O000003      C01131      P0099         4 2025-01-01 04:00:00   
3  O000004      C01096      P0006         1 2025-01-01 06:00:00   
4  O000005      C01639      P0060         2 2025-01-01 08:00:00   

   Customer_Name        City  Age  Gender  
0   Customer_861     Kolkata   51  Female  
1  Customer_1295   Bangalore   43    Male  
2  Customer_1131        Pune   47  Female  
3  Customer_1096       Delhi   29    Male  
4  Customer_1639  Coimbatore   67  Female  
Final shape: (15000, 12)
['Order_ID', 'Customer_ID', 'Product_ID', 'Quantity', 'Order_Date', 'Customer_Name', 'City', 'Age', 'Gender', 'Product_Name', 'Category', 'Price']
(15000, 12)
['Order_ID', 'Customer_ID', 'Customer_Name', 'City', 'Age', 'Gender', 'Product_ID', 'Product_Name', 'Category', 'Price', 'Qu

In [12]:
# 15. Create:
# Order_Value = Price × Quantity
final_df['Order_Value'] = final_df['Price'] * final_df['Quantity']
print(final_df[['Product_Name', 'Price', 'Quantity', 'Order_Value']].head())

# 16. Calculate total revenue.
total_revenue = final_df['Order_Value'].sum()
print(f'Total Revenue: ₹{total_revenue:,.2f}')

# 17. Calculate average order value.
avg_order_value = final_df['Order_Value'].mean()
print(f'Average Order Value: ₹{avg_order_value:,.2f}')

# 18. Find total revenue by product category.
revenue_by_category = final_df.groupby('Category')['Order_Value'].sum().sort_values(ascending=False)
print(revenue_by_category)

# 19. Find total revenue by city.
revenue_by_city = final_df.groupby('City')['Order_Value'].sum().sort_values(ascending=False)
print(revenue_by_city)

# 20. Find the top 10 customers by spending.
top10_customers = (
    final_df.groupby(['Customer_ID', 'Customer_Name'])['Order_Value']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
print(top10_customers)

# 21. Find the top 10 products by revenue.
top10_products = (
    final_df.groupby(['Product_ID', 'Product_Name'])['Order_Value']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
print(top10_products)

# 22. Find the most frequently purchased product.
purchase_freq = final_df.groupby(['Product_ID', 'Product_Name']).size().sort_values(ascending=False)
print('Most frequently purchased product:', purchase_freq.idxmax())

# 23. Find the city with the highest number of customers.
city_counts = customers['City'].value_counts()
print('City with highest number of customers:', city_counts.idxmax())

# 24. Find the city generating the highest revenue. 
revenue_by_city = final_df.groupby('City')['Order_Value'].sum().sort_values(ascending=False)
print('City with highest revenue:', revenue_by_city.idxmax())

  Product_Name     Price  Quantity  Order_Value
0   Product_27  56171.59         4    224686.36
1   Product_47  80831.23         3    242493.69
2   Product_99  83546.72         4    334186.88
3    Product_6    651.66         1       651.66
4   Product_60  81819.68         2    163639.36
Total Revenue: ₹2,233,171,871.30
Average Order Value: ₹148,878.12
Category
Sports             5.426713e+08
Clothing           4.261489e+08
Home Appliances    3.313110e+08
Electronics        3.312926e+08
Books              3.050563e+08
Beauty             2.966918e+08
Name: Order_Value, dtype: float64
City
Kolkata       3.155500e+08
Hyderabad     2.932639e+08
Coimbatore    2.831225e+08
Mumbai        2.803566e+08
Pune          2.797505e+08
Delhi         2.654541e+08
Bangalore     2.637059e+08
Chennai       2.519685e+08
Name: Order_Value, dtype: float64
Customer_ID  Customer_Name
C00019       Customer_19      3176078.87
C02897       Customer_2897    2528868.53
C01915       Customer_1915    2476932.74
C02709

In [ ]:
# 25. Divide customers into age groups.
# 18–30
# 31–45
# 46–60
# 61+
def age_group(age):
    if age <= 30:
        return '18-30'
    elif age <= 45:
        return '31-45'
    elif age <= 60:
        return '46-60'
    else:
        return '61+'

customers['Age_Group'] = customers['Age'].apply(age_group)
print(customers['Age_Group'].value_counts())

# 26. Calculate revenue generated by each age group.
revenue_by_age_group = final_df.groupby('Age_Group')['Order_Value'].sum().sort_values(ascending=False)
print(revenue_by_age_group)

# 27. Compare spending between male and female customers.
gender_comparison = final_df.groupby('Gender').agg(
    Total_Revenue=('Order_Value', 'sum'),
    Order_Count=('Order_ID', 'count'),
    Avg_Order_Value=('Order_Value', 'mean')
)
print(gender_comparison)

# 28. Identify high-value customers.
customer_spending = final_df.groupby(['Customer_ID', 'Customer_Name'])['Order_Value'].sum()

mean_spend = customer_spending.mean()
std_spend = customer_spending.std()
threshold = mean_spend + std_spend

high_value_customers = customer_spending[customer_spending > threshold].sort_values(ascending=False)
print('Number of high-value customers:', len(high_value_customers))

# 29. Determine the percentage of customers who have placed more than one order
orders_per_customer = orders.groupby('Customer_ID').size()

repeat_customers = (orders_per_customer > 1).sum()
pct_repeat_all = (repeat_customers / len(customers)) * 100

print(f'Percentage of customers with more than 1 order: {pct_repeat_all:.2f}%')

In [14]:
# 30. Create a customer spending ranking.
customer_spending = final_df.groupby(['Customer_ID', 'Customer_Name'])['Order_Value'].sum().reset_index()
customer_spending['Spending_Rank'] = customer_spending['Order_Value'].rank(ascending=False, method='min').astype(int)
customer_spending = customer_spending.sort_values('Spending_Rank')

print(customer_spending.head(10))

# 31. Identify the top 10% of customers based on total spending.
customer_spending = final_df.groupby(['Customer_ID', 'Customer_Name'])['Order_Value'].sum().reset_index()

threshold_90 = customer_spending['Order_Value'].quantile(0.90)
top_10_pct = customer_spending[customer_spending['Order_Value'] >= threshold_90].sort_values('Order_Value', ascending=False)

print('Number of customers in top 10%:', len(top_10_pct))

# 32. Find the most profitable product category for each city.
revenue_by_city_category = final_df.groupby(['City', 'Category'])['Order_Value'].sum().reset_index()

top_category_per_city = revenue_by_city_category.loc[
    revenue_by_city_category.groupby('City')['Order_Value'].idxmax()
]
print(top_category_per_city)

# 33. Find the most popular product in each category.
qty_by_category_product = final_df.groupby(['Category', 'Product_ID', 'Product_Name'])['Quantity'].sum().reset_index()

most_popular_per_category = qty_by_category_product.loc[
    qty_by_category_product.groupby('Category')['Quantity'].idxmax()
]
print(most_popular_per_category)

# 34. Calculate monthly revenue.
final_df['Year_Month'] = final_df['Order_Date'].dt.to_period('M')
monthly_revenue = final_df.groupby('Year_Month')['Order_Value'].sum()
print(monthly_revenue)

# 35. Identify the month with the highest revenue
monthly_revenue = final_df.groupby('Year_Month')['Order_Value'].sum()
print('Highest revenue month:', monthly_revenue.idxmax())
print('Revenue:', monthly_revenue.max())

     Customer_ID  Customer_Name  Order_Value  Spending_Rank
16        C00019    Customer_19   3176078.87              1
2872      C02897  Customer_2897   2528868.53              2
1896      C01915  Customer_1915   2476932.74              3
2688      C02709  Customer_2709   2400905.35              4
183       C00187   Customer_187   2394078.09              5
678       C00685   Customer_685   2352464.63              6
1283      C01297  Customer_1297   2345815.39              7
254       C00259   Customer_259   2330070.43              8
912       C00921   Customer_921   2257528.60              9
266       C00271   Customer_271   2226186.64             10
Number of customers in top 10%: 298
          City Category  Order_Value
5    Bangalore   Sports  71326795.18
11     Chennai   Sports  68298118.24
17  Coimbatore   Sports  69553033.93
23       Delhi   Sports  61019537.25
29   Hyderabad   Sports  65716665.57
35     Kolkata   Sports  71822933.47
41      Mumbai   Sports  63898188.25
47      